In [6]:
import cv2
import mediapipe as mp
import numpy as np

In [7]:
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh()

In [8]:
# Eye landmarks
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

In [9]:
# def eye_aspect_ratio(eye_points, landmarks, w, h):
#     coords = []
#     for point in eye_points:
#         x = int(landmarks[point].x * w)
#         y = int(landmarks[point].y * h)
#         coords.append((x, y))

#     vertical1 = np.linalg.norm(np.array(coords[1]) - np.array(coords[5]))
#     vertical2 = np.linalg.norm(np.array(coords[2]) - np.array(coords[4]))
#     horizontal = np.linalg.norm(np.array(coords[0]) - np.array(coords[3]))

#     return (vertical1 + vertical2) / (2.0 * horizontal)

# cap = cv2.VideoCapture(0)

# while True:
#     ret, frame = cap.read()
#     frame = cv2.flip(frame, 1)
#     h, w, _ = frame.shape

#     rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#     results = face_mesh.process(rgb)

#     if results.multi_face_landmarks:
#         for face_landmarks in results.multi_face_landmarks:
#             landmarks = face_landmarks.landmark

#             # Drowsiness detection
#             left_ear = eye_aspect_ratio(LEFT_EYE, landmarks, w, h)
#             right_ear = eye_aspect_ratio(RIGHT_EYE, landmarks, w, h)
#             ear = (left_ear + right_ear) / 2

#             if ear < 0.25:
#                 cv2.putText(frame, "DROWSY!", (50, 80),
#                             cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

#             # Distraction detection
#             nose_x = landmarks[1].x
#             if nose_x < 0.4:
#                 cv2.putText(frame, "Looking Left", (50, 130),
#                             cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
#             elif nose_x > 0.6:
#                 cv2.putText(frame, "Looking Right", (50, 130),
#                             cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

#     cv2.imshow("Smart Study Monitor", frame)

#     if cv2.waitKey(1) & 0XFF==ord('q'):
#         break

# cap.release()
# cv2.destroyAllWindows()

In [ ]:
import cv2
import mediapipe as mp
import numpy as np

# Initialize MediaPipe Face Mesh
mp_face = mp.solutions.face_mesh
face_mesh = mp_face.FaceMesh()

# Eye landmark points
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

# Function to calculate Eye Aspect Ratio (EAR)
def calculate_ear(eye_points, landmarks, width, height):
    points = []

    # Get pixel coordinates of eye points
    for p in eye_points:
        x = int(landmarks[p].x * width)
        y = int(landmarks[p].y * height)
        points.append((x, y))

    # Calculate distances
    vertical1 = np.linalg.norm(np.array(points[1]) - np.array(points[5]))
    vertical2 = np.linalg.norm(np.array(points[2]) - np.array(points[4]))
    horizontal = np.linalg.norm(np.array(points[0]) - np.array(points[3]))

    # Return EAR formula
    ear = (vertical1 + vertical2) / (2 * horizontal)
    return ear


# Start webcam
cap = cv2.VideoCapture(0)

while True:
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)  # Mirror image
    height, width, _ = frame.shape

    # Convert to RGB (MediaPipe needs RGB)
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = face_mesh.process(rgb_frame)

    if result.multi_face_landmarks:
        for face in result.multi_face_landmarks:
            landmarks = face.landmark

            # ---- Drowsiness Detection ----
            left_ear = calculate_ear(LEFT_EYE, landmarks, width, height)
            right_ear = calculate_ear(RIGHT_EYE, landmarks, width, height)

            average_ear = (left_ear + right_ear) / 2

            if average_ear < 0.25:
                cv2.putText(frame, "DROWSY!", (50, 80),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)

            # ---- Head Direction Detection ----
            nose_position = landmarks[1].x  # Nose x coordinate

            if nose_position < 0.4:
                cv2.putText(frame, "Looking Left", (50, 130),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

            elif nose_position > 0.6:
                cv2.putText(frame, "Looking Right", (50, 130),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    cv2.imshow("Smart Study Monitor", frame)
    if cv2.waitKey(1) & 0XFF==ord('q'):
        break

cap.release()
cv2.destroyAllWindows()